# ozon-geo: какие товары уходят в какие кластеры и города

**Как пользоваться:** отредактируйте параметры в разделе 1 и выполните ячейки сверху вниз (*Run All*).
Данные берутся из одного из трёх источников: `csv` (отчёты из кабинета Ozon «Заказы с моих складов»), `api` (Ozon Seller API) или `demo` (синтетика).

Установка (один раз, из корня репозитория): `pip install -e ".[notebook]"`.

In [ ]:
%matplotlib inline
import glob, os
from pathlib import Path

import pandas as pd
from IPython.display import display
import ipywidgets as W

from ozon_geo import analytics, viz
from ozon_geo.store import Store
pd.set_option("display.width", 200, "display.max_columns", 30, "display.float_format", "{:,.2f}".format)

## 1. Параметры

In [ ]:
# --- источник данных: "csv" | "api" | "demo" -------------------------------------------
SOURCE = os.environ.get("OZON_GEO_SOURCE", "csv")

# csv: маски файлов выгрузок из кабинета (можно несколько через запятую). Порядок не важен,
# повторяющиеся отправления берутся из самой свежей выгрузки.
CSV_FILES = os.environ.get("OZON_GEO_CSV", "orders/*.csv")

# api: ключи берутся из переменных окружения OZON_CLIENT_ID / OZON_API_KEY (роль «Только чтение»)
API_DAYS = 90            # за сколько дней загрузить
API_SCHEMES = ("fbs",)   # FBO: ("fbs", "fbo")

# необязательно: CSV с колонками «Артикул»;«Категория» — даёт разрез по категориям
CATEGORIES_CSV = os.environ.get("OZON_GEO_CATEGORIES", "categories.csv")

DB_PATH = os.environ.get("OZON_GEO_DB", "ozon.db")   # локальная база (SQLite)
OUT_DIR = Path(os.environ.get("OZON_GEO_OUT", "out"))
OUT_DIR.mkdir(exist_ok=True)

## 2. Загрузка данных

In [ ]:
store = Store(DB_PATH)

if SOURCE == "csv":
    from ozon_geo.importer import import_postings_files
    files = sorted({f for mask in CSV_FILES.split(",") for f in glob.glob(mask.strip())})
    if not files:
        raise SystemExit(f"Не найдено файлов по маске {CSV_FILES!r}. Положите выгрузки в папку orders/ или поменяйте CSV_FILES.")
    print("Файлы:", *files, sep="\n  ")
    import_postings_files(store, files)
elif SOURCE == "api":
    from datetime import datetime, timedelta, timezone
    from ozon_geo.client import OzonClient
    from ozon_geo.sync import sync
    now = datetime.now(timezone.utc)
    sync(OzonClient(), store, now - timedelta(days=API_DAYS), now, API_SCHEMES)
elif SOURCE == "demo":
    from ozon_geo.demo import generate
    generate(store, days=365)
else:
    raise ValueError("SOURCE должен быть csv, api или demo")

if os.path.exists(CATEGORIES_CSV):
    from ozon_geo.importer import read_category_map
    cmap = read_category_map(CATEGORIES_CSV)
    store.save_category_map(cmap)
    print(f"Категории заданы для {len(cmap)} артикулов")

sales = store.sales_frame()
print(f"\nПозиций: {len(sales):,} | отправлений: {sales.posting_number.nunique():,} | SKU: {sales.sku.nunique()} | "
      f"период: {sales.created_at.min():%Y-%m-%d} → {sales.created_at.max():%Y-%m-%d}")
print(f"Без города: {(sales.city == '(не определён)').mean():.1%} | без кластера: {(sales.cluster == '(не определён)').mean():.1%}")
months = sales.assign(месяц=sales.created_at.dt.strftime("%Y-%m")).groupby("месяц").agg(
    отправлений=("posting_number", "nunique"), штук=("quantity", "sum"), выручка=("revenue", "sum"))
display(months)
if (sales.created_at.max() - sales.created_at.min()).days < 360:
    print("⚠ История короче года: сезонность не видна, прогнозам и разрезам по SKU×кластер верьте осторожно.")

## 3. Отчёт: куда что уезжает
Выберите разрез и нажмите **Построить**. Цвет на карте — доля направления в продажах строки, число в клетке — выбранная метрика.
`lift` > 1 в таблице — направление берёт товар чаще, чем в среднем по магазину (при малых объёмах это может быть шум).

In [ ]:
dd = lambda label, opts, val: W.Dropdown(options=opts, value=val, description=label, style={"description_width": "110px"})
w_level = dd("Строки", [("SKU", "sku"), ("Артикул", "offer"), ("Категория", "category"), ("Тип товара", "type")], "sku")
w_geo = dd("Направление", [("Кластер доставки", "cluster"), ("Город доставки", "city")], "cluster")
w_metric = dd("Метрика", [("Штуки", "quantity"), ("Заказы", "orders"), ("Выручка", "revenue")], "quantity")
w_deliv = W.Checkbox(value=False, description="только доставленные", indent=False)
w_canc = W.Checkbox(value=False, description="включить отменённые", indent=False)
w_rows = W.IntSlider(value=25, min=5, max=80, description="Строк на карте", style={"description_width": "110px"})
w_cols = W.IntSlider(value=15, min=5, max=40, description="Направлений", style={"description_width": "110px"})
d0, d1 = sales.created_at.min().date(), sales.created_at.max().date()
w_dates = W.SelectionRangeSlider(options=pd.date_range(d0, d1).date.tolist(), index=(0, (d1 - d0).days),
                                 description="Период", layout=W.Layout(width="650px"), style={"description_width": "110px"})
btn = W.Button(description="Построить", button_style="primary")
out = W.Output()

def current():
    since, to = w_dates.value
    return analytics.prepare(sales, w_canc.value, since, pd.Timestamp(to) + pd.Timedelta(days=1), w_deliv.value)

def render(_=None):
    import matplotlib.pyplot as plt
    with out:
        out.clear_output(wait=True)
        df = current()
        if df.empty:
            print("Нет данных за выбранный период/фильтры"); return
        print(f"Строк в расчёте: {len(df):,}, штук: {df.quantity.sum():,}, выручка: {df.revenue.sum():,.0f}")
        top = analytics.top_cities(df, w_level.value, 3, w_metric.value, w_geo.value)
        display(top.round(2).head(40))
        plt.show(viz.heatmap(df, w_level.value, w_geo.value, w_metric.value, w_rows.value, w_cols.value))
        plt.show(viz.weekly_totals(df))

btn.on_click(render)
display(W.VBox([W.HBox([w_level, w_geo, w_metric]), W.HBox([w_deliv, w_canc]), W.HBox([w_rows, w_cols]), w_dates, btn, out]))
render()

## 4. Помесячно: SKU × кластер
Каждый столбик — месяц, каждый сегмент в нём — один SKU. SKU слишком много для палитры, поэтому **цвет — семейство товара**
(по началу названия, до 5 семейств + «Прочее»), а **оттенок внутри семейства — конкретный SKU**; белые границы делают каждый SKU отдельным куском.
Кластеры разнесены по панелям (крупнейшие + «Остальные»). Ниже — тепловая карта, где видны *все* SKU по месяцам.

In [ ]:
w_m = W.Dropdown(options=[("Штуки", "quantity"), ("Выручка", "revenue")], value="quantity", description="Метрика")
w_nc = W.IntSlider(value=7, min=2, max=11, description="Кластеров")
b3 = W.Button(description="Построить", button_style="primary")
out3 = W.Output()

def monthly(_=None):
    import matplotlib.pyplot as plt
    with out3:
        out3.clear_output(wait=True)
        df = analytics.prepare(sales, False, only_delivered=w_deliv.value)
        f1 = viz.monthly_sku_by_cluster(df, w_m.value, w_nc.value, w_geo.value)
        f1.savefig(OUT_DIR / "monthly_sku_by_cluster.png", dpi=110)
        f2 = viz.monthly_sku_heatmap(df, w_m.value)
        f2.savefig(OUT_DIR / "monthly_sku_heatmap.png", dpi=110)
        plt.show(f1); plt.show(f2)

b3.on_click(monthly)
display(W.VBox([W.HBox([w_m, w_nc, b3]), out3]))
monthly()

## 5. Базовая потребность по кластерам
Среднее за последние недели × горизонт. Это бейзлайн для планирования, а не прогноз: на малой истории и мелких SKU результат шумный.
Последняя (неполная) неделя не учитывается.

In [ ]:
w_h = W.IntSlider(value=4, min=1, max=12, description="Недель вперёд")
w_win = W.IntSlider(value=4, min=2, max=12, description="Окно, недель")
out2 = W.Output()

def plan(_=None):
    with out2:
        out2.clear_output(wait=True)
        df = analytics.prepare(sales, False, only_delivered=w_deliv.value)
        p = viz.restock_plan(df, w_level.value, w_geo.value, w_h.value, w_win.value)
        display(p.head(40))
        globals()["last_plan"] = p

b2 = W.Button(description="Рассчитать", button_style="primary"); b2.on_click(plan)
display(W.VBox([W.HBox([w_h, w_win, b2]), out2]))
plan()

## 6. Качество бейзлайна (бэктест)
Прогноз на последние недели по данным до них. WAPE — суммарная относительная ошибка (меньше — лучше).

In [ ]:
from ozon_geo import forecast
panel = analytics.weekly_panel(analytics.prepare(sales), w_level.value, w_geo.value)
panel = panel[panel["week"] < panel["week"].max()]
H = min(4, max(1, panel["week"].nunique() // 3))
pd.DataFrame({name: forecast.backtest(panel, m, H) for name, m in
              {"скользящее среднее": forecast.MovingAverage(4), "сезонный (52 нед.)": forecast.SeasonalNaive(52)}.items()})

## 7. Экспорт в Excel

In [ ]:
df = analytics.prepare(sales, False, only_delivered=w_deliv.value)
path = viz.export_excel(str(OUT_DIR / "ozon_geo.xlsx"), df, w_level.value, w_geo.value, w_h.value)
print("Сохранено:", path)